# 03 · Carga y limpieza  *(secciones 2 y 3 del script)*

**Qué hacemos:** cargamos el CSV unificado (notebook 02), lo verificamos y lo limpiamos con **8 tareas**:

1. Quitar duplicados exactos.
2. Convertir las 6 columnas de fecha de texto a `datetime` real.
3. Resolver los nulos de la categoría de producto.
4. Separar "sin comentario" de "comentario vacío" en las reseñas.
5. Validar que las coordenadas (cliente y vendedor) caigan dentro del rango de Brasil.
6. Guardar la cantidad de ítems por pedido antes de eliminar `order_item_id`.
7. Eliminar columnas que no aportan al análisis.
8. Guardar el artefacto `data/pipeline/01_df_limpio.csv`.

**Para qué:** son la base para poder calcular tiempos de entrega, agrupar por categoría, hacer NLP sobre las reseñas y graficar el mapa **sin que datos corruptos, mal tipados o columnas irrelevantes** compliquen esos cálculos más adelante.

> **Regla que gobierna toda la limpieza:** *no se inventan ni se borran datos por tener problemas.* Los nulos "reales" (fechas de pedidos que nunca se entregaron, por ejemplo) quedan en `NaN` — imputarlos sería mentir sobre el estado del negocio.

## Paso 1 — Carga y verificación

**Qué hacemos:** leemos el CSV unificado y miramos forma, tipos de dato y primeras filas.

**Para qué:** confirmar que la unificación salió bien antes de tocar nada: cantidad de filas/columnas esperada y que las columnas de las 9 tablas originales están presentes en una sola tabla.

In [1]:
# Celda estándar: imports + rutas (explicadas en 01_configuracion_inicial)
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

BASE = Path.cwd()
if not (BASE / "data").exists() and (BASE.parent / "data").exists():
    BASE = BASE.parent

CSV_UNIFICADO = BASE / "data" / "olist_dataset_unificado.csv"
PIPELINE_DIR = BASE / "data" / "pipeline"

if not CSV_UNIFICADO.exists():
    raise FileNotFoundError(
        f"No existe {CSV_UNIFICADO}. Ejecutá primero el notebook 02_unificacion_de_las_9_tablas.ipynb."
    )

df = pd.read_csv(CSV_UNIFICADO)
print(f"filas={df.shape[0]:,} columnas={df.shape[1]}")
print()
df.info()
df.head(3)

filas=112,650 columnas=39

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 39 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       112650 non-null  object 
 1   order_item_id                  112650 non-null  int64  
 2   product_id                     112650 non-null  object 
 3   seller_id                      112650 non-null  object 
 4   shipping_limit_date            112650 non-null  object 
 5   price                          112650 non-null  float64
 6   freight_value                  112650 non-null  float64
 7   customer_id                    112650 non-null  object 
 8   order_status                   112650 non-null  object 
 9   order_purchase_timestamp       112650 non-null  object 
 10  order_approved_at              112635 non-null  object 
 11  order_delivered_carrier_date   111456 non-null  object 
 12  ord

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,customer_state,payment_value_total,payment_installments_max,payment_type_principal,review_score,review_comment_message,customer_lat,customer_lng,seller_lat,seller_lng
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,RJ,72.19,2.0,credit_card,5.0,"Perfeito, produto entregue antes do combinado.",-21.762775,-41.309633,-22.496953,-44.127492
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,SP,259.83,3.0,credit_card,4.0,NaN,-20.220527,-50.903424,-23.565096,-46.518565
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,MG,216.87,5.0,credit_card,5.0,Chegou antes do prazo previsto e o produto sur...,-19.870305,-44.593326,-22.262584,-46.171124


**Insight:** la forma coincide con la que salió del notebook 02 y, en los tipos de dato, se ve el problema a resolver en el paso 3: **las 6 columnas de fecha llegan como `object` (texto)**, no como fechas. También aparecen las columnas con nulos esperadas (reviews sin comentario, fechas de pedidos no entregados, geolocalización sin match).

## Paso 2 — Duplicados exactos

**Qué hacemos:** contamos y eliminamos filas **idénticas en todas las columnas** (`drop_duplicates`).

**Para qué:** los cruces masivos pueden replicar filas si alguna tabla trae claves repetidas; una fila 100% idéntica nunca aporta información nueva y sí sesga conteos y promedios. (Ojo: ítems repetidos *dentro* de un pedido — mismo producto comprado 2 veces — **no** son duplicados: se distinguen por `order_item_id`.)

In [2]:
duplicados = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicados exactos eliminados: {duplicados}")

Duplicados exactos eliminados: 0


## Paso 3 — Fechas a tipo real

**Qué hacemos:** convertimos con `pd.to_datetime` las 6 columnas de fecha (`shipping_limit_date`, las 5 fechas del pedido). Con `errors="coerce"`, cualquier texto que no se pueda parsear queda en `NaT` (el "NaN" de las fechas) en vez de romper la ejecución.

**Para qué:** todos los cálculos del análisis son **diferencias entre fechas** (tiempo de entrega, entrega real vs. estimada, mes de compra); con texto no se puede restar una fecha de otra.

In [3]:
COLUMNAS_FECHA = [
    "shipping_limit_date", "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in COLUMNAS_FECHA:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(df[COLUMNAS_FECHA].dtypes.to_string())
print()
print("Compras desde:", df["order_purchase_timestamp"].min())
print("Compras hasta :", df["order_purchase_timestamp"].max())

shipping_limit_date              datetime64[ns]
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]

Compras desde: 2016-09-04 21:15:19
Compras hasta : 2018-09-03 09:06:57


**Insight:** las 6 columnas ahora son `datetime64[ns]` y el rango de compras corresponde al período del dataset (2016-2018). Ninguna fila se perdió: lo que no se pudo parsear (si hubiera) quedó en `NaT`, no en una excepción.

## Paso 4 — Nulos en la categoría de producto

**Qué hacemos:** rellenamos `product_category_name_english` con el literal `"sin_categoria"` y limpiamos espacios con `.str.strip()`.

**Para qué:** hay productos cuya categoría no vino o no se pudo traducir; como se reporta en todos los gráficos por categoría, conviene que aparezcan **agrupados bajo un nombre explícito** (`"sin_categoria"`) y no como un `NaN` que desaparezca de los `groupby` y `value_counts` sin aviso.

In [4]:
nulos_antes = df["product_category_name_english"].isna().sum()

df["product_category_name_english"] = (
    df["product_category_name_english"].fillna("sin_categoria").str.strip()
)

nulos_despues = df["product_category_name_english"].isna().sum()
print(f"Categorías nulas: {nulos_antes} antes -> {nulos_despues} después")
print(f"Valor de respaldo: 'sin_categoria' con {(df['product_category_name_english'] == 'sin_categoria').sum():,} filas")

Categorías nulas: 1627 antes -> 0 después
Valor de respaldo: 'sin_categoria' con 1,627 filas


## Paso 5 — Reseñas: ¿sin comentario o comentario vacío?

**Qué hacemos:** **primero** creamos la flag `tiene_comentario` (1 si el mensaje original no era nulo, 0 si era nulo) y **recién después** rellenamos los nulos con `""` (string vacío).

**Para qué:** el orden importa. Si llenáramos los nulos antes, todas las reseñas dirían "tienen texto" y perderíamos la diferencia entre:

- reseñas **sin comentario** (el cliente puso solo estrellas), y
- reseñas con **comentario vacío** (mandó un string en blanco).

Esa flag es la que después permite armar los "bolsas de texto" del NLP (notebook 09) sin arrastrar strings vacíos.

In [5]:
df["tiene_comentario"] = df["review_comment_message"].notna().astype(int)
df["review_comment_message"] = df["review_comment_message"].fillna("")

print(df["tiene_comentario"].value_counts().sort_index().to_string())
print()
print(f"% de ítems con comentario escrito: {df['tiene_comentario'].mean() * 100:.1f}%")

tiene_comentario
0    65179
1    47471



% de ítems con comentario escrito: 42.1%


**Insight:** la mayoría de las reseñas es "solo estrellas" — conviene no contarlas como texto vacío, y la flag `tiene_comentario` queda justamente para filtrarlas en el NLP.

## Paso 6 — Validación de coordenadas (cliente y vendedor)

**Qué hacemos:** para `customer` y `seller`, convertimos lat/lng a numérico y verificamos que caigan dentro del rango geográfico de Brasil (latitud entre **-34 y 6**, longitud entre **-74 y -32**). Lo que esté **fuera de rango pasa a `NaN`**.

**Para qué:** una coordenada fuera de ese rango es un dato corrupto (un zip mal geocodificado, un cero donde faltaba el valor, etc.). Un solo punto en el lugar equivocado arruinaría el mapa (notebook 08) y, sobre todo, la **distancia Haversine** (notebook 04), que daría kilómetros absurdos. Se pone en `NaN` **sin borrar la fila**: el pedido sigue siendo válido para todos los demás análisis.

In [6]:
for prefijo in ["customer", "seller"]:
    df[f"{prefijo}_lat"] = pd.to_numeric(df[f"{prefijo}_lat"], errors="coerce")
    df[f"{prefijo}_lng"] = pd.to_numeric(df[f"{prefijo}_lng"], errors="coerce")

    geo_valida = df[f"{prefijo}_lat"].between(-34, 6) & df[f"{prefijo}_lng"].between(-74, -32)
    fuera_de_rango = (~geo_valida & df[f"{prefijo}_lat"].notna()).sum()
    df.loc[~geo_valida, [f"{prefijo}_lat", f"{prefijo}_lng"]] = np.nan

    print(f"{prefijo:8s}: fuera de rango anuladas = {fuera_de_rango:6,} | "
          f"total sin coordenadas = {df[f'{prefijo}_lat'].isna().sum():,}")

customer: fuera de rango anuladas =      9 | total sin coordenadas = 311
seller  : fuera de rango anuladas =      0 | total sin coordenadas = 253


**Insight:** ninguna fila se elimina: los casos inválidos quedan en `NaN`. Los nulos totales de coordenadas combinan dos causas — zips que no matchearon contra la tabla de geolocalización (ya venían en `NaN` desde el notebook 02) y coordenadas fuera del rango de Brasil que este paso acaba de anular.

## Paso 7 — Salvar `items_por_pedido` antes de dropear `order_item_id`

**Qué hacemos:** guardamos, en la columna nueva `items_por_pedido`, cuántos ítems tiene cada pedido (`groupby("order_id")["order_item_id"].transform("count")`).

**Para qué:** `order_item_id` es el número de línea del ítem dentro del pedido; la columna siguiente elimina una lista de columnas que incluye a `order_item_id`. **Si se elimina sin guardar antes esta info**, dos unidades reales del mismo producto compradas juntas pasan a verse como "filas duplicadas" — no lo son, son ítems distintos. `transform` repite el conteo del pedido en **cada** fila de ese pedido (a diferencia de `transform("nunique")`... acá queremos el conteo de líneas).

In [7]:
df["items_por_pedido"] = df.groupby("order_id")["order_item_id"].transform("count")

print("Distribución de ítems por pedido (primeros valores):")
print(df["items_por_pedido"].value_counts().sort_index().head(8).to_string())
print()
print(f"Filas que pertenecen a pedidos con más de 1 ítem: {(df['items_por_pedido'] > 1).sum():,}")

Distribución de ítems por pedido (primeros valores):
items_por_pedido
1    88863
2    15032
3     3966
4     2020
5     1020
6     1188
7      154
8       64

Filas que pertenecen a pedidos con más de 1 ítem: 23,787


**Insight:** la inmensa mayoría de los pedidos trae un solo producto; los pedidos multi-ítem son la excepción — dato que se reutiliza en las estadísticas descriptivas (notebook 05).

## Paso 8 — Columnas que no aportan

**Qué hacemos:** eliminamos 7 columnas. Motivo de cada una:

| Columna | Por qué se elimina |
|---|---|
| `product_category_name` | ya está la versión traducida `product_category_name_english` |
| `product_name_lenght` | longitud del nombre: metadato de catálogo, no analiza |
| `product_description_lenght` | idem, descripción |
| `product_photos_qty` | cantidad de fotos: no se usa en ningún cálculo |
| `order_item_id` | su información relevante ya quedó en `items_por_pedido` |
| `customer_zip_code_prefix` | solo se usó para el join de geolocalización |
| `seller_zip_code_prefix` | idem |

**Para qué:** menos columnas = tablas y heatmaps más legibles, memoria más liviana y cero riesgo de que alguien analice por accidente una columna que solo existía para armar los cruces.

In [8]:
COLUMNAS_A_ELIMINAR = [
    "product_category_name",        # ya tenemos la traducción en product_category_name_english
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "order_item_id",                # su información relevante ya quedó en items_por_pedido
    "customer_zip_code_prefix",     # solo se usaba para el join de geolocalización
    "seller_zip_code_prefix",
]
df = df.drop(columns=[c for c in COLUMNAS_A_ELIMINAR if c in df.columns])

print(f"Filas finales: {df.shape[0]:,} | columnas finales: {df.shape[1]}")
print()
print("Columnas que quedan:")
print(list(df.columns))

Filas finales: 112,650 | columnas finales: 34

Columnas que quedan:
['order_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_city', 'seller_state', 'customer_unique_id', 'customer_city', 'customer_state', 'payment_value_total', 'payment_installments_max', 'payment_type_principal', 'review_score', 'review_comment_message', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng', 'tiene_comentario', 'items_por_pedido']


### Resumen de la limpieza

- **0 filas eliminadas**: solo se quitaron duplicados exactos (prácticamente nulos en un dataset ya consolidado); fechas y coordenadas inválidas quedan en `NaN`/`NaT`, nunca se borra un pedido por datos faltantes.
- Los pedidos que nunca llegaron a "entregado" (cancelados, en camino…) **no tienen fecha de entrega real** y eso es correcto: no corresponde inventarles una.
- Quedaron tipados correctos (`datetime`), flags útiles (`tiene_comentario`, `items_por_pedido`) y coordenadas validadas.

## Guardar el artefacto del pipeline

**Qué hacemos:** escribimos `data/pipeline/01_df_limpio.csv`.

**Para qué:** es el contrato entre este notebook y el 04 (feature engineering): el siguiente no repite la limpieza, solo carga este archivo.

> **Detalle técnico:** un CSV **no conserva tipos** (`datetime` se guarda como texto). Por eso cada notebook posterior incluye la línea estándar `pd.to_datetime` sobre las 6 columnas de fecha — es la "rehidratación" del tipo después de cargar.

In [9]:
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)
ruta_limpio = PIPELINE_DIR / "01_df_limpio.csv"
df.to_csv(ruta_limpio, index=False)
print(f"Artefacto del notebook 03 guardado -> {ruta_limpio}")
print(f"({df.shape[0]:,} filas x {df.shape[1]} columnas)")

Artefacto del notebook 03 guardado -> D:\Ciencia de Datos\Proyecto Integrador\trabajo_ integrador_versionNati\trabajo_ integrador_versionNati\data\pipeline\01_df_limpio.csv
(112,650 filas x 34 columnas)


**Siguiente paso:** `04_feature_engineering.ipynb` — a partir de esta tabla limpia, creamos las 9 variables nuevas del análisis (tiempos de entrega, envío demorado, distancia, volumen, etc.).